## 라이브러리 호출 및 데이터 로드

In [ ]:
# 라이브러리 호출
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import platform
import ast
from collections import Counter
import json
from pprint import pprint

warnings.filterwarnings('ignore')

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # Mac
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 컬럼 너비 제한 해제
pd.set_option('display.max_colwidth', None)

In [21]:
# 데이터 로드
df_champion = pd.read_csv('../../유저단위_게임데이터_상위랭커보존-explode_champion_1.csv')
df_combination = pd.read_csv('../../유저단위_게임데이터_상위랭커보존-explode_combination_1.csv')
df_champion_with_items = pd.read_csv("../../유저단위_게임데이터_상위랭커보존-explode_items_1.csv")

In [22]:
# 통계용 라이브러리 호출
from scipy import stats
import scikit_posthocs as sp

---
### **경기데이터와 챔피언 데이터가 결합된 테이블**

In [7]:
df_champion.head(2)

,gameid,user_id,user_tier,ranked,ranked_1,top4_flag,champion_name,items,star,flag_1,flag_2,name,cost,origin,class
0,KR_4291707834,KR-USER-1,platinum,5,False,False,ziggs,[7],1,0,0,ziggs,1,Rebel,Demolitionist
1,KR_4291707834,KR-USER-1,platinum,5,False,False,ashe,[9],1,0,0,ashe,3,Celestial,Sniper


#### **[1. 챔피언의 코스트에 따른 평균 등수]**

In [ ]:
# 원본 데이터에서 챔피언 정보가 있는 경우만 필터링
df_champ_analysis = df_champion[df_champion['flag_1'] == 0]

# 코스트별 평균 등수
print(df_champ_analysis.groupby('cost')['ranked'].mean()) # 평균
print('='*40)
print(df_champ_analysis.groupby('cost')['ranked'].std(ddof=1)) # 표준편차

cost
1    4.520615
2    4.557698
3    4.425134
4    4.332336
5    3.807864
Name: ranked, dtype: float64
cost
1    2.297875
2    2.298301
3    2.281277
4    2.252432
5    2.155418
Name: ranked, dtype: float64


In [31]:
alpha = 0.05 # 유의수준

# 코스트별 정규성 검정 (Shapiro-Wilk)
for cost in sorted(df_champ_analysis['cost'].unique()):
    data = df_champ_analysis[df_champ_analysis['cost'] == cost]['ranked']
    stat, p_val = stats.shapiro(data)
    
    if p_val < alpha:  # 0.05 → alpha로 변경
        result = '정규성 불만족 → Kruskal-Wallis 사용'
    else:
        result = '정규성 만족 → ANOVA 사용 가능'
    
    print(f'cost {cost}: stat={stat:.4f}, p={p_val:.4e} | {result}')

cost 1: stat=0.9250, p=1.8321e-137 | 정규성 불만족 → Kruskal-Wallis 사용
cost 2: stat=0.9247, p=8.6741e-140 | 정규성 불만족 → Kruskal-Wallis 사용
cost 3: stat=0.9266, p=5.1827e-142 | 정규성 불만족 → Kruskal-Wallis 사용
cost 4: stat=0.9290, p=6.7147e-138 | 정규성 불만족 → Kruskal-Wallis 사용
cost 5: stat=0.9222, p=1.3813e-130 | 정규성 불만족 → Kruskal-Wallis 사용


In [34]:
# 챔피언의 코스트별로 그룹화하기
groups = {}

for cost, group in df_champ_analysis.groupby('cost'):
    groups[cost] = group['ranked']

# Kruskal-Wallis 검정 통계량, p-vlaue
h_stat, kw_p = stats.kruskal(groups[1], groups[2], groups[3], groups[4], groups[5])

# η²_H 효과크기 계산
k5=5 # 집단수
n5=len(groups[1])+len(groups[2])+len(groups[3])+len(groups[4])+len(groups[5]) # 표본의 크기
eta_sq_h5 = (h_stat - k5 + 1) / (n5 - k5)

if eta_sq_h5 is not None:
    size_eta3 = '작은' if eta_sq_h5 < 0.06 else '중간' if eta_sq_h5 < 0.14 else '큰'
    print(f"[효과크기] η²_H = {eta_sq_h5:.4f} ({size_eta3} 효과)")

print(f'\nKruskal-Wallis: h_stat={h_stat:.4f}, p={kw_p:.4e}')

if kw_p < alpha:
    print('\n코스트별 평균 등수 차이 유의미함 → Dunn 사후검정 진행')
else:
    print('\n코스트별 평균 등수 차이 유의미하지 않음')

[효과크기] η²_H = 0.0094 (작은 효과)

Kruskal-Wallis: h_stat=29360.2726, p=0.0000e+00

코스트별 평균 등수 차이 유의미함 → Dunn 사후검정 진행


In [ ]:
# Dunn's 사후검정
dunn_result = sp.posthoc_dunn(
    df_champ_analysis,
    val_col='ranked',       # 비교할 값
    group_col='cost',       # 그룹을 나누는 기준
    p_adjust='fdr_bh'       # Benjamini-Hochberg 보정 적용
)

# 코스트별 중앙값, 평균 미리 계산
# -> 비모수 검정이므로 중앙값으로 방향 해석, 중앙값이 같으면 평균으로 보조 해석
medians = {}
means = {}

for cost, group in df_champ_analysis.groupby('cost'):
    medians[cost] = group['ranked'].median()
    means[cost] = group['ranked'].mean()

# 쌍별 결과 해석
cost_names = dunn_result.columns.tolist()

for i in range(len(cost_names)):
    for j in range(i + 1, len(cost_names)):
        c1, c2 = cost_names[i], cost_names[j]
        p_val = dunn_result.iloc[i, j]
        sig = '유의미한 차이 o' if p_val < alpha else '유의미한 차이 x'

        m1, m2 = medians[c1], medians[c2]
        if m1 < m2:
            direction = f'{c1}코스트 중앙값({m1:.1f})이 {c2}코스트({m2:.1f})보다 등수가 높습니다'
        elif m1 > m2:
            direction = f'{c2}코스트 중앙값({m2:.1f})이 {c1}코스트({m1:.1f})보다 등수가 높습니다'
        else:
            # 중앙값이 같으면 평균으로도 비교
            a1, a2 = means[c1], means[c2]
            if a1 < a2:
                direction = f'중앙값 동일({m1:.1f}), 평균 기준 {c1}코스트({a1:.2f})가 {c2}코스트({a2:.2f})보다 등수 높음'
            elif a1 > a2:
                direction = f'중앙값 동일({m1:.1f}), 평균 기준 {c2}코스트({a2:.2f})가 {c1}코스트({a1:.2f})보다 등수 높음'
            else:
                direction = f'중앙값, 평균 모두 동일 ({m1:.1f})'

        print(f'{c1}코스트 vs {c2}코스트: {sig} → {direction}')
        print()

1코스트 vs 2코스트: 유의미한 차이 o → 중앙값 동일(5.0), 평균 기준 1코스트(4.52)가 2코스트(4.56)보다 등수 높음

1코스트 vs 3코스트: 유의미한 차이 o → 3코스트 중앙값(4.0)이 1코스트(5.0)보다 등수가 높습니다

1코스트 vs 4코스트: 유의미한 차이 o → 4코스트 중앙값(4.0)이 1코스트(5.0)보다 등수가 높습니다

1코스트 vs 5코스트: 유의미한 차이 o → 5코스트 중앙값(4.0)이 1코스트(5.0)보다 등수가 높습니다

2코스트 vs 3코스트: 유의미한 차이 o → 3코스트 중앙값(4.0)이 2코스트(5.0)보다 등수가 높습니다

2코스트 vs 4코스트: 유의미한 차이 o → 4코스트 중앙값(4.0)이 2코스트(5.0)보다 등수가 높습니다

2코스트 vs 5코스트: 유의미한 차이 o → 5코스트 중앙값(4.0)이 2코스트(5.0)보다 등수가 높습니다

3코스트 vs 4코스트: 유의미한 차이 o → 중앙값 동일(4.0), 평균 기준 4코스트(4.33)가 3코스트(4.43)보다 등수 높음

3코스트 vs 5코스트: 유의미한 차이 o → 중앙값 동일(4.0), 평균 기준 5코스트(3.81)가 3코스트(4.43)보다 등수 높음

4코스트 vs 5코스트: 유의미한 차이 o → 중앙값 동일(4.0), 평균 기준 5코스트(3.81)가 4코스트(4.33)보다 등수 높음



##### 뭔가 독특한 점은..대부분 역시 예상대로 코스트가 높을수록 평균 등수나 중앙값이 낮은 편인데, 1,2코스트는 뒤바뀐 느낌이 듦
-> 게임단위에서도 같은 경향이 보이는지 확인해볼 필요는 있을 것 같음

---
### **경기데이터와 콤비네이션 데이터가 결합된 테이블**

In [48]:
df_combination.head(2)

,gameid,user_id,user_tier,ranked,top4_flag,ranked_1,flag_1,flag_2,active_synergies,synergy,synergy_val
0,KR_4291707834,KR-USER-1,platinum,5,False,False,0,0,{},Cybernetic,1
1,KR_4291707834,KR-USER-1,platinum,5,False,False,0,0,{},Demolitionist,1


In [54]:
# active_synergies가 빈 딕셔너리면 0, 아니면 1
df_combination['has_active'] = df_combination['active_synergies'].apply(
    lambda x: 0 if ast.literal_eval(x) == {} else 1
)

In [ ]:
# gameId + user_id 기준으로 중복 제거 → 한 게임에서 한 유저 = 하나의 관측치
df_combo_user = df_combination.drop_duplicates(subset=['gameid', 'user_id'])

# 교차표 생성
ct = pd.crosstab(df_combo_user['has_active'], df_combo_user['top4_flag'])
print(ct)
print()

# 카이제곱 독립성 검정
# 귀무가설: 활성화된 조합 여부와 top4 여부는 독립적이다.
# 대립가설: 활성화된 조합 여부와 top4 여부는 독립적이지 않다.
chi2, p_val, dof, expected = stats.chi2_contingency(ct)
print(f'chi2={chi2:.4f}, p={p_val:.4e}')
print()

if p_val < alpha:
    print('활성화된 조합 여부가 top4에 유의미한 영향을 줌')
else:
    print('활성화된 조합 여부가 top4에 유의미한 영향을 주지 않음')

top4_flag    False   True 
has_active                
0              926     282
1           197152  197879

chi2=343.5760, p=1.0619e-76

활성화된 조합 여부가 top4에 유의미한 영향을 줌


In [ ]:
# Cramér's V
n = df_combo_user.shape[0]  # 전체 샘플 수
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1))) # min(ct.shape): 행, 열 중 작은 값이 필요한 효과크기 공식이라서 작성함
print(f"Cramér's V: {cramers_v:.4f}")

Cramér's V: 0.0294


In [61]:
total = len(df_combo_user)

no_active_ratio = (df_combo_user['has_active'] == 0).sum() / total
no_active_top4_ratio = (
    (df_combo_user['has_active'] == 0) & (df_combo_user['top4_flag'] == True)
).sum() / total  # has_active == 0 으로 수정
active_top4_ratio = (
    (df_combo_user['has_active'] == 1) & (df_combo_user['top4_flag'] == True)
).sum() / total

print(f'활성화된 조합이 없는 유저의 비율: {no_active_ratio:.2%}')
print(f'활성화된 조합이 없는데, top4에 속한 유저의 비율: {no_active_top4_ratio:.2%}')
print(f'활성화된 조합이 있는데, top4에 속한 유저의 비율: {active_top4_ratio:.2%}')

활성화된 조합이 없는 유저의 비율: 0.30%
활성화된 조합이 없는데, top4에 속한 유저의 비율: 0.07%
활성화된 조합이 있는데, top4에 속한 유저의 비율: 49.94%


사실상 대부분의 유저가 조합을 활성화된 상태에서 게임을 하고, 조합을 활성화하지 못하면 top 4에 들기는 매우 어렵다

---

In [64]:
# active_synergies 문자열 자체를 고유값으로 카운트
unique_combos = df_combo_user['active_synergies'].nunique()
print(f'고유한 조합 구성 수: {unique_combos}')

고유한 조합 구성 수: 16587


In [65]:
# 가장 많이 활성화된 조합 구성 Top 5
top_combos = df_combo_user['active_synergies'].value_counts().head(5)
print(top_combos)

active_synergies
{'Blaster': 4, 'Chrono': 2, 'Mercenary': 1, 'Brawler': 4}                                                        24296
{'Blaster': 4, 'Chrono': 2, 'Mercenary': 1, 'Rebel': 3, 'Brawler': 4, 'Starship': 1}                              9313
{'Blaster': 4, 'Chrono': 2, 'Brawler': 4}                                                                         7218
{'Blaster': 2, 'Chrono': 4, 'ManaReaver': 2, 'Mercenary': 1, 'Blademaster': 3, 'Celestial': 2, 'Valkyrie': 2}     7111
{'Mystic': 2, 'Sorcerer': 4, 'StarGuardian': 6}                                                                   6981
Name: count, dtype: int64


In [67]:
# 전체 데이터에서 Top 5 조합이 차지하는 비율
top_combos_ratio = df_combo_user['active_synergies'].value_counts(normalize=True).head(5)
print(top_combos_ratio.map(lambda x: f'{x:.2%}'))

active_synergies
{'Blaster': 4, 'Chrono': 2, 'Mercenary': 1, 'Brawler': 4}                                                        6.13%
{'Blaster': 4, 'Chrono': 2, 'Mercenary': 1, 'Rebel': 3, 'Brawler': 4, 'Starship': 1}                             2.35%
{'Blaster': 4, 'Chrono': 2, 'Brawler': 4}                                                                        1.82%
{'Blaster': 2, 'Chrono': 4, 'ManaReaver': 2, 'Mercenary': 1, 'Blademaster': 3, 'Celestial': 2, 'Valkyrie': 2}    1.79%
{'Mystic': 2, 'Sorcerer': 4, 'StarGuardian': 6}                                                                  1.76%
Name: proportion, dtype: str


가장 많이 쓰이는 조합 Top 3에는 Blaster', 'Chrono', 'Brawler'가 모두 포함되어 있다.

---
### **경기데이터와 챔피언 데이터, 아이템 데이터가 결합된 테이블**

In [15]:
# df_champion_with_items.head(2)